[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/motion_straight_line.ipynb)

# Motion in a straight line full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step), **tier 2** core problems (the three
equations, free fall, graphs, relative velocity) and **tier 3** stretch problems (combine ideas, spot the trap, run an
equation backwards). Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same physics in a few lines of plain Python
and numpy, so you can try your own numbers. And the course's simulations, to play with: the strips under a
velocity–time graph, a ball thrown straight up (change the **throw speed** and watch the arch grow), and the van
from the start of the course (change its **speed**, the driver's **reaction time** and the **deceleration**, and
watch where it stops). Every
simulation steps the motion forward a thousandth of a second at a time, and checks itself against the formula.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the equation and the steps.
Two decimals are enough, in the units the question asks for (you can type the unit too: `17.32 m/s`). A minus sign
means the other direction: `-10`. You can type the arithmetic too: `sqrt(2*5*30)`, `72/3.6`. For several answers,
separate them with commas, in the order the question lists them (the box's label shows the order). You can also put
your answer between the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight
away.

**The cheat sheet.** Every equation and graph rule from the course on two pages:
[motion in a straight line cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/motion_straight_line_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and positions in chalk, chalk smudge, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# bodies, positions and distances CHALK · velocity SKY · acceleration MINT · force CORAL · energy VIOLET ·
# the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})
FOOT_SIM = "simulated · checked against the formula"
FOOT_OWN = "an invented example · our own numbers"

def footnote(fig, text="our own numbers · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

G = 9.8                                               # m/s², g near the ground (the course's value)
G_MOON = 1.62                                         # m/s², the Moon's (NASA's Moon fact sheet)

# ---- the simulator: the course's own, a fixed-step fourth-order Runge–Kutta ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """Step the motion forward from t = 0 to t_end in steps of dt. deriv(t, s) gives the rates of change of the state s
    (here s = (x, v), so the rates are (v, a)). Returns the list of (t, state) samples. With stop(t, s), the run ends at
    the first step where stop is True, placed on the event by interpolation (stop.var = the state's index to watch,
    stop.level = the level it crosses)."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s2 = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t2 = t + dt
        if stop is not None and stop(t2, s2):
            i = getattr(stop, "var", None)
            if i is not None and s2[i] != s[i]:
                f = (getattr(stop, "level", 0.0) - s[i]) / (s2[i] - s[i])
                t2 = t + f * dt
                s2 = tuple(a + f * (b - a) for a, b in zip(s, s2))
            out.append((t2, s2))
            return out
        t, s = t2, s2
        out.append((t, s))
    return out

def crossing(var, level=0.0):
    """A stop test for simulate: the run ends when state[var] falls below level."""
    f = lambda t, s: s[var] < level
    f.var, f.level = var, level
    return f

def close(a, b, rel=1e-4):
    """A simulated value matches the formula (to a relative 10⁻⁴, as in the course)."""
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), 1e-9)

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"(?<![A-Za-z])(m/s²|m/s\^2|m/s2|m/s|km/h|kmh|metres|meters|metre|meter|seconds|second|secs|sec|s|m)(?![A-Za-z/²^0-9])"

def read_part(s):
    s = re.sub(UNITS, "", s.strip()).strip()
    s = s.replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**").replace("√", "sqrt")
    try: return calc(s)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the course's practice.py) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Position and displacement',
     "q": 'A dog walks 30 m east from a lamp post, then 40 m west. What path length does it cover, in metres?',
     "parts": ['path length (m)'],
     "answer": [70.0], "tol": [0.005],
     "hint": 'path length: add every step',
     "working": '30 + 40 = 70 m (path length adds every step, whatever its direction)'},
    {"id": 'P1.2', "tier": 1, "group": 'Position and displacement',
     "q": "Same walk, with east positive and the lamp post as origin: what is the dog's displacement, in metres?",
     "parts": ['displacement (m)'],
     "answer": [-10.0], "tol": [0.005],
     "hint": 'displacement = final − initial position',
     "working": 'final − initial position = (30 − 40) − 0 = −10 m: 10 m west of the post'},
    {"id": 'P1.3', "tier": 1, "group": 'Velocity',
     "q": 'A swimmer swims one 50 m length of a pool and back in 80 s. What is her average speed, in m/s?',
     "parts": ['average speed (m/s)'],
     "answer": [1.25], "tol": [0.005],
     "hint": 'average speed = path length ÷ time',
     "working": 'path length / time = 100/80 = 1.25 m/s'},
    {"id": 'P1.4', "tier": 1, "group": 'Velocity',
     "q": 'Same swim: what is her average velocity, in m/s?',
     "parts": ['average velocity (m/s)'],
     "answer": [0.0], "tol": [0.005],
     "hint": 'average velocity = displacement ÷ time',
     "working": 'displacement / time = 0/80 = 0: she ends where she started'},
    {"id": 'P1.5', "tier": 1, "group": 'Velocity',
     "q": 'Convert 72 km/h to metres per second.',
     "parts": ['speed (m/s)'],
     "answer": [20.0], "tol": [0.005],
     "hint": 'm/s = km/h ÷ 3.6',
     "working": '72/3.6 = 20 m/s'},
    {"id": 'P1.6', "tier": 1, "group": 'Acceleration',
     "q": 'An e-scooter goes from rest to 6 m/s in 4 s. What is its average acceleration, in m/s²?',
     "parts": ['acceleration (m/s²)'],
     "answer": [1.5], "tol": [0.005],
     "hint": 'a = Δv ÷ Δt',
     "working": 'Δv/Δt = (6 − 0)/4 = 1.5 m/s²'},
    {"id": 'P1.7', "tier": 1, "group": 'Acceleration',
     "q": 'A bus moves at a steady 12 m/s for 5 s. Using the area under its velocity–time graph, how far does it go, in metres?',
     "parts": ['distance (m)'],
     "answer": [60.0], "tol": [0.005],
     "hint": 'distance = area under v–t (a rectangle)',
     "working": 'a rectangle: 12 × 5 = 60 m'},
    {"id": 'P1.8', "tier": 1, "group": 'Equations of motion',
     "q": 'A cyclist at 3 m/s speeds up at 2 m/s² for 4 s. What is her velocity at the end, in m/s?',
     "parts": ['velocity (m/s)'],
     "answer": [11.0], "tol": [0.005],
     "hint": 'v = u + a t',
     "working": 'v = u + a t = 3 + 2 × 4 = 11 m/s'},
    {"id": 'P1.9', "tier": 1, "group": 'Free fall',
     "q": 'A stone is dropped from rest. Ignoring air, how fast is it falling after 2 s, in m/s?',
     "parts": ['speed (m/s)'],
     "answer": [19.6], "tol": [0.005],
     "hint": 'v = g t, from rest',
     "working": 'v = g t = 9.8 × 2 = 19.6 m/s'},
    {"id": 'P1.10', "tier": 1, "group": 'Relative velocity',
     "q": "Car A drives at 22 m/s and car B at 30 m/s, the same way along a straight road. What is B's velocity relative to A, in m/s?",
     "parts": ['B relative to A (m/s)'],
     "answer": [8.0], "tol": [0.005],
     "hint": 'v_BA = v_B − v_A',
     "working": 'v_BA = v_B − v_A = 30 − 22 = 8 m/s'},
    {"id": 'P2.1', "tier": 2, "group": 'Acceleration',
     "q": 'A train slows from 25 m/s to 10 m/s in 30 s. What is its average acceleration, in m/s²?',
     "parts": ['acceleration (m/s²)'],
     "answer": [-0.5], "tol": [0.005],
     "hint": 'a = (v − u) ÷ t',
     "working": '(10 − 25)/30 = −0.5 m/s²: the minus sign points against the motion'},
    {"id": 'P2.2', "tier": 2, "group": 'Equations of motion',
     "q": 'A car at 15 m/s accelerates at 1.5 m/s² for 6 s. How far does it go in that time, in metres?',
     "parts": ['distance (m)'],
     "answer": [117.0], "tol": [0.005],
     "hint": 'x = u t + ½ a t²',
     "working": 'x = u t + ½ a t² = 90 + 27 = 117 m'},
    {"id": 'P2.3', "tier": 2, "group": 'Equations of motion',
     "q": 'A car at 25 m/s brakes at 6 m/s² until it stops. What is its braking distance, in metres?',
     "parts": ['braking distance (m)'],
     "answer": [52.083333333333336], "tol": [0.005],
     "hint": 'v² = u² + 2 a x, with v = 0',
     "working": '0 = 25² − 2 × 6 × x → x = 625/12 = 52.08 m'},
    {"id": 'P2.4', "tier": 2, "group": 'Equations of motion',
     "q": 'A motorbike starts from rest and reaches 30 m/s after 225 m at a steady acceleration. What is the acceleration, in m/s²?',
     "parts": ['acceleration (m/s²)'],
     "answer": [2.0], "tol": [0.005],
     "hint": 'v² = u² + 2 a x, with u = 0',
     "working": 'v² = 2 a x → a = 900/450 = 2 m/s²'},
    {"id": 'P2.5', "tier": 2, "group": 'Calculus',
     "q": "A runner's position is x = 2t² (metres, seconds). What is her velocity at t = 3 s, in m/s?",
     "parts": ['velocity at 3 s (m/s)'],
     "answer": [12.0], "tol": [0.005],
     "hint": 'v = dx/dt',
     "working": 'v = dx/dt = 4t = 12 m/s at 3 s'},
    {"id": 'P2.6', "tier": 2, "group": 'Free fall',
     "q": 'A ball is thrown straight up at 19.6 m/s. How long does it take to reach the top, and how high does it rise (s, m)?',
     "parts": ['time to the top (s)', 'height (m)'],
     "answer": [2.0, 19.6], "tol": [0.005, 0.005],
     "hint": 'v = 0 at the top: t = u ÷ g; h = u² ÷ 2g',
     "working": 't = u/g = 2 s; h = u²/(2g) = 384.16/19.6 = 19.6 m'},
    {"id": 'P2.7', "tier": 2, "group": 'Free fall',
     "q": 'In a ruler-drop test, the ruler falls 18 cm before you catch it. What is your reaction time, in s?',
     "parts": ['reaction time (s)'],
     "answer": [0.19166296949998196], "tol": [0.005],
     "hint": 'd = ½ g t²',
     "working": 'd = ½ g t² → t = √(2 × 0.18/9.8) = 0.192 s'},
    {"id": 'P2.8', "tier": 2, "group": 'Graphs',
     "q": "A tram's velocity–time graph: 0 to 8 m/s in 4 s, steady 8 m/s for 6 s, then 8 m/s to 0 in 2 s. How far does it go, in metres?",
     "parts": ['distance (m)'],
     "answer": [72.0], "tol": [0.005],
     "hint": 'area under v–t: triangles and a rectangle',
     "working": 'areas: ½ × 4 × 8 = 16, 8 × 6 = 48, ½ × 2 × 8 = 8; total 72 m'},
    {"id": 'P2.9', "tier": 2, "group": 'Graphs',
     "q": 'A drone moving up at 1 m/s has an acceleration of 2 m/s² for 3 s, then none. What is its velocity afterwards, in m/s?',
     "parts": ['velocity afterwards (m/s)'],
     "answer": [7.0], "tol": [0.005],
     "hint": 'change in v = area under a–t',
     "working": 'area under the acceleration–time graph = 2 × 3 = 6 m/s of change: 1 + 6 = 7 m/s'},
    {"id": 'P2.10', "tier": 2, "group": 'Relative velocity',
     "q": 'Two cars 700 m apart drive towards each other at 20 m/s and 15 m/s. After how many seconds do they meet?',
     "parts": ['time to meet (s)'],
     "answer": [20.0], "tol": [0.005],
     "hint": 'closing speed = the sum of the two speeds',
     "working": 'relative speed 20 + 15 = 35 m/s; 700/35 = 20 s'},
    {"id": 'P3.1', "tier": 3, "group": 'Velocity',
     "q": 'You drive 60 km at 60 km/h and come back at 40 km/h. What is your average speed for the round trip, in km/h? (It is not 50.)',
     "parts": ['average speed (km/h)'],
     "answer": [48.0], "tol": [0.005],
     "hint": 'average speed = total distance ÷ total time',
     "working": 'times 1 h and 1.5 h; 120 km / 2.5 h = 48 km/h (more time is spent at the slower speed)'},
    {"id": 'P3.2', "tier": 3, "group": 'Acceleration',
     "q": 'A cart moves at +5 m/s for 4 s, then at −3 m/s for 6 s. Give its displacement and its path length (m).',
     "parts": ['displacement (m)', 'path length (m)'],
     "answer": [2.0, 38.0], "tol": [0.005, 0.005],
     "hint": 'areas below the axis count negative',
     "working": 'areas: +20 above the axis, −18 below; displacement 2 m, path length 38 m'},
    {"id": 'P3.3', "tier": 3, "group": 'Equations of motion',
     "q": 'A car stops in 30 m when braking at 5 m/s². How fast was it going, in m/s? (Run the third equation backwards.)',
     "parts": ['speed (m/s)'],
     "answer": [17.320508075688775], "tol": [0.005],
     "hint": 'v² = u² + 2 a x, run backwards',
     "working": '0 = u² − 2 × 5 × 30 → u = √300 = 17.32 m/s'},
    {"id": 'P3.4', "tier": 3, "group": 'Equations of motion',
     "q": 'A car accelerates from rest at 2 m/s² for 5 s, then brakes at 4 m/s² to a stop. How far does it go in all, in metres?',
     "parts": ['total distance (m)'],
     "answer": [37.5], "tol": [0.005],
     "hint": 'one equation per constant-acceleration phase',
     "working": 'phase 1: 25 m and 10 m/s; phase 2: 10²/(2 × 4) = 12.5 m; total 37.5 m (one equation per constant-acceleration phase)'},
    {"id": 'P3.5', "tier": 3, "group": 'Calculus',
     "q": "A cart's position is x = 3t² − 0.5t³ (m, s). Give its velocity at t = 2 s, and the time after start when it stops for a moment (m/s, s).",
     "parts": ['velocity at 2 s (m/s)', 'time when it stops (s)'],
     "answer": [6.0, 4.0], "tol": [0.005, 0.005],
     "hint": 'v = dx/dt; it stops where v = 0',
     "working": 'v = 6t − 1.5t²: 12 − 6 = 6 m/s at 2 s; v = 0 at t = 6/1.5 = 4 s'},
    {"id": 'P3.6', "tier": 3, "group": 'Free fall',
     "q": 'A stone is thrown straight down at 5 m/s from a 20 m bridge. How long until it hits the water, in s?',
     "parts": ['time (s)'],
     "answer": [1.5735283527206263], "tol": [0.005],
     "hint": 'x = u t + ½ g t², a quadratic in t',
     "working": '20 = 5t + 4.9t² → t = (−5 + √(25 + 392))/9.8 = 1.574 s'},
    {"id": 'P3.7', "tier": 3, "group": 'Free fall',
     "q": 'The same 10 m/s upward throw on Earth and on the Moon (g = 1.62 m/s²): how high does it rise in each (m)?',
     "parts": ['height on Earth (m)', 'height on the Moon (m)'],
     "answer": [5.1020408163265305, 30.864197530864196], "tol": [0.005, 0.005],
     "hint": 'h = u² ÷ 2g',
     "working": 'h = u²/(2g): Earth 5.10 m, Moon 30.86 m (g is about six times smaller)'},
    {"id": 'P3.8', "tier": 3, "group": 'Relative velocity',
     "q": 'A car at 30 m/s is behind a truck at 22 m/s and must gain 60 m to pass it. How long does the overtake take, and how far does the car travel meanwhile (s, m)?',
     "parts": ['time (s)', "the car's distance (m)"],
     "answer": [7.5, 225.0], "tol": [0.005, 0.005],
     "hint": 'time = gap ÷ relative velocity',
     "working": 'relative velocity 8 m/s; 60/8 = 7.5 s; the car covers 30 × 7.5 = 225 m'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(f"The equation: {q['hint']}.", 110))
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes in any form, a wrong one doesn't
assert right(read_part("17.32 m/s"), math.sqrt(300), 0.005) and right(read_part("sqrt(2*5*30)"), math.sqrt(300), 0.005)
assert right(read_part("−10 m"), -10, 0.005) and right(read_part("72/3.6"), 20, 0.005) and not right(read_part("17.3"), math.sqrt(300), 0.005)
assert right(read_part("2 m/s²"), 2, 0.005) and right(read_part("48 km/h"), 48, 0.005) and right(read_part("1.5 s"), 1.5, 0.005)

## Tier 1 · Warm-up

One idea, one step: path length and displacement, average speed and average velocity, kilometres an hour to metres a
second, acceleration from a change of velocity, a distance as an area, the first equation, falling from rest, and one
velocity relative to another.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

The course's equations are one line each, so they make good Python functions: `v_at(u, a, t)` is v = u + a t,
`x_at(u, a, t)` is x = u t + ½ a t², `brake_dist(u, a)` is u² ÷ 2a, and so on. The cells below use them. The rest of
the warm-ups are one line of arithmetic each; each line computes one warm-up answer and checks it against the practice
set. Directions are signs: east (or up) is positive, so 10 m west is −10.

In [ ]:
def v_at(u, a, t): return u + a * t                                 # v = u + a t
def x_at(u, a, t): return u * t + 0.5 * a * t * t                   # x = u t + ½ a t²
def v_after(u, a, x): return math.sqrt(u * u + 2 * a * x)            # v² = u² + 2 a x (the forward root)
def brake_dist(u, a): return u * u / (2 * a)                         # from u to rest at deceleration a
def stop_dist(u, tr, a): return u * tr + brake_dist(u, a)            # reaction distance + braking distance
def fall_time(h, g=G): return math.sqrt(2 * h / g)                   # h = ½ g t², from rest
def ms(kmh): return kmh / 3.6                                        # km/h → m/s
def kmh(v): return v * 3.6                                           # m/s → km/h

path = 30 + 40                                                       # P1.1: path length adds every step
disp = (30 - 40) - 0                                                 # P1.2: final − initial position
swim_speed, swim_vel = (50 + 50) / 80, (50 - 50) / 80                # P1.3, P1.4: back where she started
print(f"dog: path {path} m, displacement {disp} m   swimmer: speed {swim_speed} m/s, velocity {swim_vel} m/s")
print(f"72 km/h = {ms(72)} m/s   scooter a = {(6 - 0) / 4} m/s²   bus: 12 × 5 = {12 * 5} m   cyclist v = "
      f"{v_at(3, 2, 4)} m/s   stone after 2 s: {G * 2:.1f} m/s   B relative to A: {30 - 22} m/s")
for k, v in (("P1.1", [path]), ("P1.2", [disp]), ("P1.3", [swim_speed]), ("P1.4", [swim_vel]), ("P1.5", [ms(72)]),
             ("P1.6", [(6 - 0) / 4]), ("P1.7", [12 * 5]), ("P1.8", [v_at(3, 2, 4)]), ("P1.9", [G * 2]),
             ("P1.10", [30 - 22])):
    assert agrees(k, v), k
assert v_at(10, 2, 5) == 20 and x_at(10, 2, 5) == 75 and 10 ** 2 + 2 * 2 * 75 == 20 ** 2   # the course's car: 400 = 400

### Strips under a velocity–time graph

A car speeds up steadily from rest to 10 m/s in 4 s. How far does it go? The distance is the area under its
velocity–time graph: a triangle, ½ × 4 × 10 = 20 m. Cut the time into strips: each strip is a thin rectangle, its
height the speed at the strip's left edge, its width the strip's time. With 4 strips the rectangles add to 15 m, with
40 to 19.5 m, with 400 to 19.95 m: more strips, closer to the triangle. Slide the number of strips.

In [ ]:
def strip_sum(n, top=10.0, T=4.0):
    """The strips' areas added: n rectangles under v = (top/T) t, heights at each strip's left edge (numpy)."""
    w = T / n
    left = np.arange(n) * w                         # each strip's left edge (s)
    return float((top / T * left * w).sum())        # speed × width, added

def strips_picture(strips=4):
    top, T = 10.0, 4.0
    w = T / strips; left = np.arange(strips) * w
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.18)
    ax.bar(left, top / T * left, width=w, align="edge", color=CHALK, alpha=0.25, edgecolor=CHALK,
           linewidth=0.8 if strips <= 60 else 0)
    ax.plot([0, T], [0, top], color=SKY)
    ax.set_xlim(0, T * 1.02); ax.set_ylim(0, top * 1.1); ax.set_xlabel("time (s)"); ax.set_ylabel("velocity (m/s)")
    fig.suptitle(f"{strips} strips: {strip_sum(strips):.2f} m   ·   the triangle: ½ × {T:g} × {top:g} = {0.5 * T * top:g} m",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert [round(strip_sum(n), 2) for n in (4, 40, 400)] == [15, 19.5, 19.95]                 # the video's numbers

In [ ]:
interact(strips_picture, strips=IntSlider(value=4, min=1, max=400, step=1, description="strips"));

## Tier 2 · Core

The course's standard problems, several steps each: an acceleration from two velocities, the three equations
forwards and backwards, velocity as the slope dx/dt, a ball thrown up, the ruler-drop reaction test, a distance from a
velocity–time graph, a velocity from an acceleration–time graph, and two cars closing on each other.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**The equations** are the functions from tier 1. **A velocity at an instant** is the slope of x–t: shrink the window
and watch the average velocity close in (for the car pulling away, x = 1.5 t², from 2 s: 7.5, 6.15, 6.015 → 6 m/s).
**A graph read both ways**: the area under v–t is the distance (numpy adds the trapezia), and running sums rebuild v
and x from an acceleration–time graph: the course's lift goes up 2 + 20 + 2 = 24 m, eight floors of 3 m.

In [ ]:
assert agrees("P2.1", [(10 - 25) / 30]) and agrees("P2.2", [x_at(15, 1.5, 6)]) and agrees("P2.3", [brake_dist(25, 6)])
assert agrees("P2.4", [30 ** 2 / (2 * 225)])                                               # v² = 2 a x, backwards
assert agrees("P2.6", [19.6 / G, brake_dist(19.6, G)]) and agrees("P2.7", [fall_time(0.18)])
assert agrees("P2.9", [1 + 2 * 3]) and agrees("P2.10", [700 / (20 + 15)])

x = lambda t: 2 * t ** 2                                                                   # P2.5: v = dx/dt
windows = [(x(3 + h) - x(3)) / h for h in (1, 0.1, 0.01, 0.001)]
print("P2.5, average velocity over shrinking windows from 3 s:", [round(w, 4) for w in windows], "→ 4t = 12 m/s")
assert agrees("P2.5", [4 * 3])
car = lambda t: 1.5 * t ** 2                                                               # the course's car
assert [round((car(2 + h) - car(2)) / h, 3) for h in (1, 0.1, 0.01)] == [7.5, 6.15, 6.015]

phases = np.array([(4, 0, 8), (6, 8, 8), (2, 8, 0)], float)                                # P2.8: (duration, v from, v to)
areas = phases[:, 0] * (phases[:, 1] + phases[:, 2]) / 2                                   # each phase's trapezium
print("P2.8, the tram: areas", areas, "→", areas.sum(), "m")
assert agrees("P2.8", [areas.sum()])

dt = 0.5                                                                                   # the lift, every 0.5 s
t = np.arange(0, 14 + dt, dt)
a = np.select([t < 2, t < 12, t < 14], [1.0, 0.0, -1.0], 0.0)                              # its acceleration–time graph
v = np.concatenate([[0.0], np.cumsum(a[:-1] * dt)])                                        # v: a running sum of a × dt
xs = np.concatenate([[0.0], np.cumsum((v[:-1] + v[1:]) / 2 * dt)])                        # x: a running sum of v × dt
print("the lift: x at 2, 12 and 14 s:", xs[t == 2][0], xs[t == 12][0], xs[t == 14][0], "m; top speed", v.max(), "m/s")
assert xs[t == 2][0] == 2 and xs[t == 12][0] == 22 and xs[-1] == 24 and xs[-1] / 3 == 8 and v[t == 7][0] == 2

### Throw it up: watch the arch grow

A ball thrown straight up at 14.7 m/s. Gravity takes 9.8 m/s off its velocity every second, so it stops rising after
14.7 ÷ 9.8 = 1.5 s, at a height of u² ÷ 2g = 11.025 m, and lands at 3 s, falling at 14.7 m/s. The simulator steps the
throw forward a thousandth of a second at a time; the title compares it with the formulas. Change the **throw speed**:
double it and the arch goes four times as high.

In [ ]:
def throw(u, g=G):
    """The simulated throw: to the top (where v crosses 0) and to the landing (where x crosses 0)."""
    to_top = simulate(lambda t, s: (s[1], -g), (0.0, u), 2 * u / g + 1, stop=crossing(1))
    flight = simulate(lambda t, s: (s[1], -g), (1e-12, u), 2 * u / g + 1, stop=crossing(0))
    return to_top[-1], flight

def throw_picture(throw_speed=14.7):
    u = throw_speed
    (t_top, (h_top, _)), flight = throw(u)
    t_land = flight[-1][0]
    assert close(t_top, u / G) and close(h_top, u * u / (2 * G)) and close(t_land, 2 * u / G)   # checked every time
    ts = np.array([t for t, _ in flight]); hs = np.array([s[0] for _, s in flight]); vs = np.array([s[1] for _, s in flight])
    every = slice(0, None, 250)                                                     # a dot every 0.25 s
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [3, 2]})
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.2, wspace=0.28)
    ax.plot(ts, hs, color=CHALK, lw=1.6); ax.scatter(ts[every], hs[every], s=22, color=CHALK, zorder=3)
    ax.scatter([t_top], [h_top], s=110, color=GOLD, zorder=4)
    ax.annotate(f"top: {t_top:.2f} s, {h_top:.3f} m", (t_top, h_top), textcoords="offset points", xytext=(8, 8), color=GOLD)
    ax.set_xlim(0, 6.3); ax.set_ylim(0, 48); ax.set_xlabel("time (s)"); ax.set_ylabel("height (m)")
    bx.plot(ts, vs, color=SKY); bx.axhline(0, color=DIM, lw=1)
    bx.set_xlim(0, 6.3); bx.set_ylim(-31, 31); bx.set_xlabel("time (s)"); bx.set_ylabel("velocity (m/s)")
    fig.suptitle(f"u = {u:g} m/s: top at {t_top:.2f} s (u/g = {u / G:.2f}), {h_top:.3f} m (u²/2g = {u * u / (2 * G):.3f}); "
                 f"lands at {t_land:.2f} s", x=0.07, ha="left", fontsize=14, color=GOLD)
    footnote(fig, FOOT_SIM); plt.show()

(_t, (_h, _)), _f = throw(14.7)                                                            # the video's numbers
assert round(_t, 6) == 1.5 and round(_h, 3) == 11.025 and round(_f[-1][0], 6) == 3 and round(-_f[-1][1][1], 6) == 14.7
assert round(throw(29.4)[0][1][0] / _h, 6) == 4                                            # double the speed: four times as high

In [ ]:
interact(throw_picture, throw_speed=FloatSlider(value=14.7, min=2, max=30, step=0.1, description="throw speed"));

## Tier 3 · Stretch

Combine ideas, choose the method, spot the trap, run it backwards: an average speed that isn't the average of the
speeds, displacement against path length, a speed from a stopping distance, two phases with two accelerations, a
moment when a cart stops, a stone thrown down, the same throw on the Moon, and an overtake.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

The traps are worth a line each. An average speed is total distance over total time, not the average of the speeds.
Areas below the time axis count negative for displacement, positive for path length (`abs`). A stopping distance run
backwards is a square root. Two accelerations mean two phases: one equation each, the second starting where the
first ends. A quadratic in t has two roots; only the positive one is a time after the throw.

In [ ]:
avg = (60 + 60) / (60 / 60 + 60 / 40)                                                      # P3.1: 48, not 50
v_t = [(5, 4), (-3, 6)]                                                                    # P3.2: (velocity, time)
assert agrees("P3.1", [avg]) and agrees("P3.2", [sum(v * t for v, t in v_t), sum(abs(v) * t for v, t in v_t)])
assert agrees("P3.3", [math.sqrt(2 * 5 * 30)])                                             # 0 = u² − 2 a x
v1 = v_at(0, 2, 5)                                                                         # P3.4: phase 1 ends at 10 m/s
assert agrees("P3.4", [x_at(0, 2, 5) + brake_dist(v1, 4)])
assert agrees("P3.5", [6 * 2 - 1.5 * 2 ** 2, 6 / 1.5])                                     # v = 6t − 1.5t²
roots = np.roots([0.5 * G, 5, -20])                                                        # P3.6: 4.9t² + 5t − 20 = 0
print("P3.6, the two roots:", np.round(roots, 3), "→ the positive one is the time")
assert agrees("P3.6", [max(roots)])
assert agrees("P3.7", [brake_dist(10, G), brake_dist(10, G_MOON)])                          # the same u²/2g
assert agrees("P3.8", [60 / (30 - 22), 30 * 60 / (30 - 22)])                               # relative velocity 8 m/s

## The van from the start: can it stop?

The course opens with a van at **50 km/h** and a child stepping out **25 m** ahead, and answers it at the end of
"Reaction time and stopping distance". A child stepping out is a surprise, so we allow **1.5 s** to react; the brakes
give **6.5 m/s²**. While the driver reacts, the van rolls on at full speed (a = 0); then it brakes at a steady
deceleration until it is at rest. The simulator runs one phase after the other. Change the van's **speed**, the
**reaction time** and the **deceleration** and watch where it stops: before the crossing, or not. At 50 km/h it needs
35.67 m and reaches the crossing still at about 42 km/h; at 30 km/h it stops in 17.84 m.

In [ ]:
VAN = {'kmh': 50.0, 'slow_kmh': 30.0, 'tr': 1.5, 'a': 6.5, 'sight': 25.0}   # the course's numbers (an invented example)

def van_stop(speed_kmh=50.0, reaction=1.5, braking=6.5):
    """The simulated stop: (t, (x, v)) samples, reacting for `reaction` s, then braking at `braking` m/s² to rest."""
    u = ms(speed_kmh)
    react = simulate(lambda t, s: (s[1], 0.0), (0.0, u), reaction)                     # phase 1: a = 0
    t0, s0 = react[-1]
    brake = simulate(lambda t, s: (s[1], -braking), s0, u / braking + 1, stop=crossing(1))   # phase 2: to rest
    return react + [(t0 + t, s) for t, s in brake[1:]]

def at_crossing(run, sight=VAN["sight"]):
    """The van's speed (m/s) as it reaches the crossing, or None when it stops first."""
    xs, vs = np.array([s[0] for _, s in run]), np.array([s[1] for _, s in run])
    return None if xs[-1] <= sight else float(np.interp(sight, xs, vs))

def van_picture(speed_kmh=50.0, reaction=1.5, braking=6.5):
    run = van_stop(speed_kmh, reaction, braking)
    u, total = ms(speed_kmh), run[-1][1][0]
    assert close(total, stop_dist(u, reaction, braking))                                 # the simulation = the sum
    ts = np.array([t for t, _ in run]); xs = np.array([s[0] for _, s in run]); vs = np.array([s[1] for _, s in run])
    hit, sight = at_crossing(run), VAN["sight"]
    fig, (ax, bx) = plt.subplots(2, 1, figsize=(11.5, 5.6), gridspec_kw={"height_ratios": [1, 2]})
    fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.15, hspace=0.55)
    rd = u * reaction
    ax.barh(0, rd, height=0.5, color=CHALK, alpha=0.30); ax.barh(0, total - rd, left=rd, height=0.5, color=CHALK, alpha=0.65)
    ax.text(rd / 2, 0.45, f"react {rd:.1f} m", ha="center", va="bottom", color=DIM, fontsize=11)
    ax.text(rd + (total - rd) / 2, 0.45, f"brake {total - rd:.1f} m", ha="center", va="bottom", color=DIM, fontsize=11)
    ax.scatter(xs[::250], np.zeros_like(xs[::250]), s=14, color=CHALK, zorder=3)        # the van every 0.25 s
    ax.axvline(sight, color=DIM, ls="--", lw=1.6); ax.text(sight, -0.75, "crossing (25 m)", ha="center", color=DIM, fontsize=11)
    if hit is None: ax.scatter([total], [0], s=120, color=GOLD, zorder=4)
    else: ax.scatter([sight], [0], s=120, color=GOLD, zorder=4)
    ax.set_xlim(0, max(75, total * 1.05)); ax.set_ylim(-1, 1.1); ax.set_yticks([]); ax.grid(False); ax.set_xlabel("distance along the street (m)")
    bx.plot(ts, vs, color=SKY); bx.fill_between(ts, vs, color=CHALK, alpha=0.12, lw=0)
    bx.set_xlim(0, max(8, ts[-1] * 1.05)); bx.set_ylim(0, 21); bx.set_xlabel("time (s)"); bx.set_ylabel("velocity (m/s)")
    verdict = (f"stops in {total:.2f} m: {sight - total:.2f} m to spare" if hit is None else
               f"needs {total:.2f} m: reaches the crossing at {kmh(hit):.1f} km/h")
    fig.suptitle(f"{speed_kmh:g} km/h, {reaction:g} s, {braking:g} m/s²: {verdict}", x=0.08, ha="left", fontsize=15,
                 color=GOLD)
    footnote(fig, FOOT_SIM + " · " + FOOT_OWN); plt.show()

assert round(van_stop(50)[-1][1][0], 2) == 35.67 and round(van_stop(30)[-1][1][0], 2) == 17.84      # the video's numbers
assert round(kmh(at_crossing(van_stop(50))), 2) == 42.40 and at_crossing(van_stop(30)) is None

In [ ]:
interact(van_picture, speed_kmh=FloatSlider(value=50, min=20, max=70, step=1, description="van speed (km/h)"),
         reaction=FloatSlider(value=1.5, min=0.5, max=2.5, step=0.1, description="reaction (s)"),
         braking=FloatSlider(value=6.5, min=2, max=9, step=0.5, description="braking (m/s²)"));

## Sources
Every exercise and number here is our own (the van's brake figure is an invented value in a realistic range). The
physics is the standard kinematics of a first course: motion in a straight line with constant acceleration, free fall without air, velocity–time and acceleration–time graphs, relative velocity.
g = 9.8 m/s² near the ground; the Moon's 1.62 m/s² is from NASA's Moon Fact Sheet
(nssdc.gsfc.nasa.gov/planetary/factsheet/moonfact.html, retrieved 2026-11-11). The 1.5 s reaction time is our
assumption for a surprise; it is in line with M. Green, "How long does it take to stop?", Transportation Human Factors
2(3), 2000 (about 1.5 s for a surprise; retrieved 2026-11-11).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph02_straight_line/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph02_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["tier"], _q["group"]) == \
               (_e["q"], _e["working"], _e["tier"], _e["group"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and \
               all(abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX) and _P.VAN == VAN and _P.G == G and _P.G_MOON == G_MOON
    assert abs(van_stop(_P.VAN["kmh"])[-1][1][0] - _P.VS["stop50"]) < 1e-3
    assert abs(van_stop(_P.VAN["slow_kmh"])[-1][1][0] - _P.VS["stop30"]) < 1e-3
    print("matches the course's practice set")